# Task 3 - Transfer Learning para Detección de Neumonía

**Curso:** CC3182 - Visión por Computadora  
**Laboratorio:** 7  

**Estudiante:** Andy Fuentes 22944

**Estudiante:** Diederich Solis  22952

**Estudiante:** Christian Echeverria 221441

**Fecha:** 26 de marzo del 2026  


## Objetivo
Implementar y comparar dos modelos preentrenados en ImageNet para clasificar radiografías de tórax en las clases **Normal** y **Neumonía**, evaluando su desempeño predictivo, costo computacional y viabilidad de uso en un entorno on-premise sin GPU.

## 1. Introducción

En este notebook se desarrolla una solución de clasificación binaria para radiografías de tórax usando **transfer learning**.  
Se comparan dos modelos preentrenados en ImageNet bajo las mismas condiciones de entrenamiento, con el fin de analizar no solo su precisión, sino también métricas más relevantes en contexto médico, como **F1-score** y **sensibilidad para la clase Neumonía**, además de su tamaño en disco y tiempo de inferencia.

In [1]:
import os
import time
import random
import zipfile
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report

## 2. Reproducibilidad

Para hacer el experimento reproducible, se fijó una semilla aleatoria para Python, NumPy y PyTorch.

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 2. Descarga del dataset desde Kaggle

Para descargar el dataset en Colab se utiliza la API de Kaggle mediante el archivo `kaggle.json`.

In [6]:
from google.colab import files
files.upload()

Saving kaggle1.json to kaggle1.json


{'kaggle1.json': b'{"username":"andyfer0040w0w","key":"cba1047b3931466d257da0ae777279b2"}'}

In [9]:
!mkdir -p ~/.kaggle
!cp kaggle1.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle1.json

In [10]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:18<00:00, 137MB/s]



In [11]:
!unzip -q chest-xray-pneumonia.zip -d /content/

In [12]:
base_dir = Path("/content/chest_xray")
print(base_dir.exists())
print(list(base_dir.iterdir()))

True
[PosixPath('/content/chest_xray/chest_xray'), PosixPath('/content/chest_xray/__MACOSX'), PosixPath('/content/chest_xray/train'), PosixPath('/content/chest_xray/test'), PosixPath('/content/chest_xray/val')]


## 4. Exploración inicial del dataset

Se revisó la estructura del dataset y la cantidad de imágenes por clase para entender la distribución inicial de los datos.

In [13]:
image_paths = []
labels = []

for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        class_dir = base_dir / split / cls
        if class_dir.exists():
            for img_path in class_dir.glob("*.*"):
                image_paths.append(str(img_path))
                labels.append(0 if cls == "NORMAL" else 1)

df = pd.DataFrame({
    "path": image_paths,
    "label": labels
})

df["label_name"] = df["label"].map({0: "NORMAL", 1: "PNEUMONIA"})
print(df["label_name"].value_counts())
df.head()

label_name
PNEUMONIA    4273
NORMAL       1583
Name: count, dtype: int64


,path,label,label_name
0,/content/chest_xray/train/NORMAL/NORMAL2-IM-06...,0,NORMAL
1,/content/chest_xray/train/NORMAL/NORMAL2-IM-07...,0,NORMAL
2,/content/chest_xray/train/NORMAL/NORMAL2-IM-06...,0,NORMAL
3,/content/chest_xray/train/NORMAL/NORMAL2-IM-04...,0,NORMAL
4,/content/chest_xray/train/NORMAL/IM-0379-0001....,0,NORMAL


## 5. Preparación del dataset

Aunque el dataset original ya incluye carpetas de entrenamiento, validación y prueba, en este trabajo se reconstruyó la partición completa para obtener una división de 70% entrenamiento, 15% validación y 15% prueba.

In [14]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 4099
Validation: 878
Test: 879


## 6. Justificación de la división

La división se realizó con estratificación para mantener una proporción similar de imágenes normales y con neumonía en los tres subconjuntos. Esto es importante porque evita sesgos en la evaluación y hace la comparación entre modelos más confiable.

In [15]:
print(train_df["label_name"].value_counts(normalize=True))
print(val_df["label_name"].value_counts(normalize=True))
print(test_df["label_name"].value_counts(normalize=True))

label_name
PNEUMONIA    0.72969
NORMAL       0.27031
Name: proportion, dtype: float64
label_name
PNEUMONIA    0.730068
NORMAL       0.269932
Name: proportion, dtype: float64
label_name
PNEUMONIA    0.729238
NORMAL       0.270762
Name: proportion, dtype: float64


## 7. Data augmentation

Para entrenamiento se aplicaron transformaciones suaves y apropiadas para imágenes médicas, con el objetivo de mejorar la generalización sin alterar de forma irreal la anatomía observada. En validación y prueba solo se aplicó resize y normalización, ya que estos conjuntos deben reflejar el comportamiento real del modelo sobre datos no alterados aleatoriamente.

In [16]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(degrees=7),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## 8. Construcción de datasets y dataloaders

Se definió una clase personalizada para cargar las imágenes y aplicar diferentes transformaciones según el subconjunto. Luego se construyeron los dataloaders de entrenamiento, validación y prueba.

In [17]:
class XRayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, "path"]
        label = self.dataframe.loc[idx, "label"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [18]:
train_dataset = XRayDataset(train_df, transform=train_transform)
val_dataset = XRayDataset(val_df, transform=eval_transform)
test_dataset = XRayDataset(test_df, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 9. Selección de modelos preentrenados

Se seleccionaron dos modelos preentrenados en ImageNet: **ResNet18** y **DenseNet121**. La comparación busca contrastar un modelo más liviano frente a uno más profundo.

## 10. Adaptación del cabezal de clasificación

En ambos modelos se congelaron las capas base y se reemplazó el cabezal final para adaptarlo al problema binario Normal vs. Neumonía. Se decidió congelar las capas convolucionales base para aprovechar características generales ya aprendidas y reducir el riesgo de sobreajuste.

In [19]:
def build_resnet18():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 1)
    )
    return model

def build_densenet121():
    model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 1)
    )
    return model

## 11. Configuración de entrenamiento

Ambos modelos se entrenaron con la misma función de pérdida y el mismo optimizador para asegurar una comparación justa. Se utilizó BCEWithLogitsLoss, Adam, learning rate de 1e-3, batch size de 32 y un máximo de 15 épocas.

In [20]:
criterion = nn.BCEWithLogitsLoss()
LR = 1e-3
EPOCHS = 15
PATIENCE = 3

## 12. Early Stopping

Se implementó Early Stopping monitoreando la pérdida de validación. La lógica consiste en detener el entrenamiento cuando el modelo deja de mejorar durante varias épocas consecutivas, reduciendo así el sobreajuste.

In [21]:
def train_model(model, train_loader, val_loader, model_name):
    model = model.to(device)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

    best_val_loss = float("inf")
    patience_counter = 0
    history = {"train_loss": [], "val_loss": []}
    best_path = f"/content/{model_name}_best.pth"

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_losses = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.float().unsqueeze(1).to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)
                val_losses.append(loss.item())

        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)

        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)

        print(f"{model_name} | Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping activado para {model_name}")
            break

    model.load_state_dict(torch.load(best_path))
    return model, history, best_path

## 13. Entrenamiento de los modelos

A continuación se entrenan los dos modelos seleccionados bajo la misma configuración.

In [22]:
resnet18 = build_resnet18()
densenet121 = build_densenet121()

resnet18, resnet_history, resnet_path = train_model(resnet18, train_loader, val_loader, "resnet18")
densenet121, densenet_history, densenet_path = train_model(densenet121, train_loader, val_loader, "densenet121")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 88.9MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 163MB/s]


resnet18 | Epoch 1/15 | Train Loss: 0.3149 | Val Loss: 0.1887
resnet18 | Epoch 2/15 | Train Loss: 0.2164 | Val Loss: 0.1704
resnet18 | Epoch 3/15 | Train Loss: 0.2043 | Val Loss: 0.2299
resnet18 | Epoch 4/15 | Train Loss: 0.1864 | Val Loss: 0.1675
resnet18 | Epoch 5/15 | Train Loss: 0.2120 | Val Loss: 0.2851
resnet18 | Epoch 6/15 | Train Loss: 0.2346 | Val Loss: 0.2118
resnet18 | Epoch 7/15 | Train Loss: 0.1771 | Val Loss: 0.1585
resnet18 | Epoch 8/15 | Train Loss: 0.1885 | Val Loss: 0.1576
resnet18 | Epoch 9/15 | Train Loss: 0.1844 | Val Loss: 0.1622
resnet18 | Epoch 10/15 | Train Loss: 0.1797 | Val Loss: 0.1869
resnet18 | Epoch 11/15 | Train Loss: 0.1772 | Val Loss: 0.1548
resnet18 | Epoch 12/15 | Train Loss: 0.1740 | Val Loss: 0.1602
resnet18 | Epoch 13/15 | Train Loss: 0.1792 | Val Loss: 0.1516
resnet18 | Epoch 14/15 | Train Loss: 0.1592 | Val Loss: 0.1539
resnet18 | Epoch 15/15 | Train Loss: 0.1718 | Val Loss: 0.1996
densenet121 | Epoch 1/15 | Train Loss: 0.2991 | Val Loss: 0.2146

## 14. Evaluación en el conjunto de prueba

Para cada modelo se calcularon las métricas solicitadas: accuracy, F1-score para la clase Neumonía, sensibilidad/recall, tamaño del modelo en disco y tiempo de inferencia promedio sobre 100 imágenes.

In [23]:
def evaluate_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).int().cpu().numpy().flatten()

            y_pred.extend(preds)
            y_true.extend(labels.numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, pos_label=1)
    recall = recall_score(y_true, y_pred, pos_label=1)

    return acc, f1, recall

In [24]:
resnet_acc, resnet_f1, resnet_recall = evaluate_model(resnet18, test_loader)
dense_acc, dense_f1, dense_recall = evaluate_model(densenet121, test_loader)

print("ResNet18:", resnet_acc, resnet_f1, resnet_recall)
print("DenseNet121:", dense_acc, dense_f1, dense_recall)

ResNet18: 0.949943117178612 0.9654088050314465 0.9578783151326054
DenseNet121: 0.9431171786120591 0.9603803486529319 0.9453978159126365


## 15. Tamaño del modelo en disco

Se midió el tamaño de cada modelo guardado en disco, expresado en megabytes, para considerar su viabilidad en entornos con recursos limitados.

In [25]:
def get_model_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

resnet_size = get_model_size_mb(resnet_path)
dense_size = get_model_size_mb(dense_path) if 'dense_path' in locals() else get_model_size_mb(densenet_path)

print("ResNet18 size (MB):", resnet_size)
print("DenseNet121 size (MB):", dense_size)

ResNet18 size (MB): 42.96296977996826
DenseNet121 size (MB): 27.61185073852539


In [26]:
dense_size = get_model_size_mb(densenet_path)

print("ResNet18 size (MB):", resnet_size)
print("DenseNet121 size (MB):", dense_size)

ResNet18 size (MB): 42.96296977996826
DenseNet121 size (MB): 27.61185073852539


## 16. Tiempo de inferencia

Se calculó el tiempo de inferencia promedio sobre 100 imágenes para estimar el costo computacional de cada modelo en un entorno real.

In [27]:
def measure_inference_time(model, dataset, n_samples=100):
    model.eval()
    indices = np.random.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False)
    times = []

    with torch.no_grad():
        for idx in indices:
            image, _ = dataset[idx]
            image = image.unsqueeze(0).to(device)

            start = time.time()
            _ = model(image)
            end = time.time()

            times.append((end - start) * 1000)

    return np.mean(times)

In [28]:
resnet_time = measure_inference_time(resnet18, test_dataset, 100)
dense_time = measure_inference_time(densenet121, test_dataset, 100)

print("ResNet18 avg inference time (ms):", resnet_time)
print("DenseNet121 avg inference time (ms):", dense_time)

ResNet18 avg inference time (ms): 2.7257513999938965
DenseNet121 avg inference time (ms): 17.959046363830566


## 17. Tabla comparativa final

Se resume el desempeño de ambos modelos en una tabla comparativa con todas las métricas solicitadas.

In [29]:
results_df = pd.DataFrame({
    "Modelo": ["ResNet18", "DenseNet121"],
    "Accuracy": [resnet_acc, dense_acc],
    "F1 Neumonía": [resnet_f1, dense_f1],
    "Recall Neumonía": [resnet_recall, dense_recall],
    "Tamaño (MB)": [resnet_size, dense_size],
    "Inferencia promedio (ms)": [resnet_time, dense_time]
})

results_df

,Modelo,Accuracy,F1 Neumonía,Recall Neumonía,Tamaño (MB),Inferencia promedio (ms)
0,ResNet18,0.949943,0.965409,0.957878,42.962970,2.725751
1,DenseNet121,0.943117,0.960380,0.945398,27.611851,17.959046


## 18. Relación entre tamaño del modelo y desempeño

Además del rendimiento predictivo, se analiza cuánto almacenamiento cuesta cada punto porcentual de F1-score para valorar si el modelo más pesado realmente justifica su costo.

In [30]:
results_df["F1 (%)"] = results_df["F1 Neumonía"] * 100
results_df["MB por punto de F1"] = results_df["Tamaño (MB)"] / results_df["F1 (%)"]

results_df[["Modelo", "Tamaño (MB)", "F1 (%)", "MB por punto de F1"]]

,Modelo,Tamaño (MB),F1 (%),MB por punto de F1
0,ResNet18,42.962970,96.540881,0.445024
1,DenseNet121,27.611851,96.038035,0.287510


## 19. Interpretación del tiempo de inferencia

El tiempo de inferencia no debe analizarse solo como un número en milisegundos, sino también en términos de su impacto dentro de un flujo clínico real. Por ello, además de reportar el tiempo promedio por imagen, se estima cuántas radiografías por hora podría procesar cada modelo de forma aproximada.

Este análisis permite valorar si el tiempo de respuesta es aceptable para un sistema de apoyo al diagnóstico en un entorno clínico, especialmente en escenarios on-premise sin GPU.

In [31]:
results_df["Imgs por segundo"] = (1000 / results_df["Inferencia promedio (ms)"]).round(2)
results_df["Imgs por hora"] = (results_df["Imgs por segundo"] * 3600).round(0)

results_df[["Modelo", "Inferencia promedio (ms)", "Imgs por segundo", "Imgs por hora"]]

,Modelo,Inferencia promedio (ms),Imgs por segundo,Imgs por hora
0,ResNet18,2.725751,366.87,1320732.0
1,DenseNet121,17.959046,55.68,200448.0


## 20. Dictamen ejecutivo

En este experimento se compararon los modelos **ResNet18** y **DenseNet121** para la detección de neumonía en radiografías de tórax, evaluando no solo su desempeño predictivo, sino también su costo computacional y su viabilidad de despliegue en un entorno on-premise sin GPU.

De acuerdo con los resultados obtenidos, **ResNet18** alcanzó un accuracy de **0.9499**, un F1-score para la clase Neumonía de **0.9654**, una sensibilidad de **0.9579**, un tamaño de **42.96 MB** y un tiempo de inferencia promedio de **2.73 ms**. Por su parte, **DenseNet121** obtuvo un accuracy de **0.9431**, un F1-score de **0.9604**, una sensibilidad de **0.9454**, un tamaño de **27.61 MB** y un tiempo de inferencia promedio de **17.96 ms**.

Desde el punto de vista clínico, la métrica más importante en este problema no es únicamente el accuracy, sino la **sensibilidad para la clase Neumonía**, ya que un falso negativo implica no detectar a un paciente posiblemente enfermo. Por esta razón, el modelo que presente mayor sensibilidad puede ser preferible, incluso si sacrifica un poco de accuracy global. En un contexto médico, suele ser más grave dejar pasar un caso positivo que revisar un falso positivo adicional.

Al comparar ambos modelos, el que detecta mejor neumonía es **ResNet18**, ya que obtuvo la mayor sensibilidad (**0.9579**) frente a **0.9454** de DenseNet121. Además, también superó a DenseNet121 en accuracy y F1-score, por lo que en este experimento mostró un mejor rendimiento general.

Sin embargo, la decisión final no depende solo de sensibilidad y accuracy. También debe considerarse el tamaño del modelo y su velocidad de inferencia. En este experimento, el costo en almacenamiento por punto porcentual de F1 fue de **0.4450 MB** para **ResNet18** y de **0.2875 MB** para **DenseNet121**. Esto indica que, aunque ResNet18 obtuvo mejores métricas, DenseNet121 es más eficiente en términos de almacenamiento por punto de F1.

En clínicas rurales o centros con infraestructura reducida, este punto es especialmente importante. Si la mejora en F1 del modelo más pesado es pequeña, puede no valer la pena asumir un mayor consumo de memoria. En este caso, ResNet18 mejora a DenseNet121 en F1 por aproximadamente **0.50 puntos porcentuales**, pero requiere alrededor de **15.35 MB** adicionales. Desde una perspectiva estrictamente de almacenamiento, DenseNet121 resulta más eficiente. Sin embargo, esa diferencia de peso debe analizarse junto con la velocidad y la sensibilidad.

Respecto al tiempo de inferencia, ambos modelos muestran tiempos suficientemente bajos para ser considerados viables en un flujo clínico real de apoyo al diagnóstico. Con base en el promedio medido, **ResNet18** podría procesar aproximadamente **1,320,732 radiografías por hora** y **DenseNet121** alrededor de **200,448 radiografías por hora**, al menos desde una estimación teórica. Aunque estos valores son ideales y en la práctica existirán otros cuellos de botella como lectura de archivos, transferencia de datos, interfaz y validación médica, los tiempos observados muestran que la inferencia del modelo no sería el factor limitante del sistema.

La recomendación final al CTO, pensando en una solución **on-premise sin GPU**, sería **ResNet18**. Esta recomendación se basa en que ofrece el mejor balance entre desempeño clínico y operativo: obtuvo la mejor sensibilidad, el mejor accuracy, el mejor F1-score y además fue considerablemente más rápido en inferencia. Aunque es más pesado que DenseNet121, su tamaño de **42.96 MB** sigue siendo manejable para un despliegue local, y la diferencia de velocidad frente a DenseNet121 es lo suficientemente amplia como para justificar su elección.

Una limitación importante del experimento es que el dataset utilizado proviene de una fuente específica y no representa toda la variabilidad posible de equipos radiológicos, protocolos de adquisición, hospitales o poblaciones. Por ello, no se puede asumir que el modelo funcionará exactamente igual con radiografías tomadas en distintos hospitales o con pacientes de características diferentes.

Esto tiene implicaciones directas para **MediScan Guatemala**. Antes de llevar cualquiera de estos modelos a producción, sería necesario validarlo con datos locales, provenientes de distintos hospitales, equipos y poblaciones, para verificar que su desempeño se mantenga estable. De lo contrario, existe el riesgo de que el modelo generalice bien en el dataset de prueba, pero falle cuando se enfrente a condiciones reales no vistas durante el entrenamiento.

Finalmente, congelar las capas base permitió aprovechar el conocimiento previo de ImageNet y entrenar de forma más eficiente con un dataset relativamente limitado. En este experimento, esta estrategia fue suficiente para obtener resultados altos en ambos modelos. Es posible que descongelar parte de las capas mejore aún más la adaptación al dominio médico, pero también aumentaría el riesgo de sobreajuste y el costo de entrenamiento. Esto se relaciona con lo discutido en el Task 2: una solución técnicamente fuerte no depende solo del modelo, sino también de cómo se adapta, cómo se valida y bajo qué condiciones puede considerarse confiable para apoyar decisiones reales.

## 21. Conclusión

En este trabajo se implementó un enfoque de transfer learning para clasificar radiografías de tórax en las clases **Normal** y **Neumonía**, comparando dos modelos preentrenados en ImageNet bajo las mismas condiciones de entrenamiento.

Los resultados mostraron que **ResNet18** fue el modelo con mejor desempeño general, alcanzando un **accuracy de 0.9499**, un **F1-score de 0.9654** y una **sensibilidad de 0.9579** para la clase Neumonía. Además, presentó un tiempo de inferencia promedio mucho menor que DenseNet121, lo cual fortalece su viabilidad para un entorno de despliegue sin GPU.

Por otro lado, **DenseNet121** tuvo la ventaja de ser más liviano en disco y más eficiente en términos de MB por punto porcentual de F1, lo que muestra que el mejor modelo no siempre es el más pequeño ni el más pesado, sino el que logra un mejor equilibrio entre rendimiento clínico, velocidad y costo computacional.

En un problema clínico como este, la sensibilidad tiene un peso especial porque disminuir los falsos negativos ayuda a no pasar por alto pacientes con neumonía. Por esa razón, aunque el accuracy sea importante, no debe ser la única métrica considerada al momento de seleccionar un modelo para producción.

Finalmente, este experimento también deja claro que un buen resultado en el dataset de prueba no garantiza el mismo comportamiento en radiografías provenientes de otros hospitales, equipos o poblaciones. Por ello, antes de considerar una implementación real para MediScan Guatemala, sería necesario realizar validación adicional con datos locales y definir cuidadosamente el contexto operativo en el que el modelo será utilizado.